# Algorithm primer: hybrid first-/second-quantization

Walkthrough: switch between first- and second-quantization representations mid-circuit.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the planner picks the route (cpu / cpu+sim / cpu+gpu) at preflight time.

## Background

**Algorithm:** hybrid quantization. First-quantization uses real-space grid wavefunctions (kinetic operator = discrete Laplacian); second-quantization uses Pauli decompositions of fermionic operators. The conversion is a specialized circuit. SDK status: research-grade, exposed via apply_linear + expv composition.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
# Minimal hybrid-quantization sketch: build a 1-D grid Laplacian
# (first-quant kinetic operator) and apply it to a state. The full
# scheme alternates conversion gates between this representation and
# the 2-quant Pauli string list.
from uniqx.core import types
from uniqx.domains.physics.kernels import grid_laplacian
from uniqx.ops import shape
from uniqx.ops.primitives.evolution import apply_linear

nx = 8
n = nx
dx = 1.0 / (nx + 1)

@trace
def prog(state):
    l_flat = grid_laplacian(nx=nx, ny=1, nz=1, dx=dx, dy=1.0, dz=1.0)
    l_mat = shape.reshape(l_flat, result_type=types.tensor("f64", [n, n]), shape=[n, n])
    return apply_linear(l_mat, state)

state = np.ones(n) / np.sqrt(n)
module = prog(state.tolist())


## Preflight — see what routes the planner found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the planner's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the planner picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
